In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!git clone https://github.com/Rania0497/vlm-legers-quantification.git
%cd vlm-legers-quantification
!ls src

Cloning into 'vlm-legers-quantification'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 135 (delta 55), reused 105 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 156.17 KiB | 9.19 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/vlm-legers-quantification
benchmark.py	    download_commons.py      make_examples.py
caption_dataset.py  evaluate_clip.py	     make_report_table.py
captioning.py	    evaluate_consistency.py  prepare_lora_images.py
check_models.py     evaluate_text.py	     split_lora.py
clean_captions.py   evaluate_uncertainty.py


In [ ]:
!mkdir -p data/raw/lora_512
!unzip -q -o /content/drive/MyDrive/vlm_lora/lora_512.zip -d data/raw/lora_512
!ls data/raw/lora_512 | wc -l
!python -m src.split_lora

30
lora_train : 25 images
lora_test : 5 images


In [ ]:
!pip install -q diffusers peft accelerate datasets

In [ ]:
import diffusers, peft, transformers, torch
print("diffusers", diffusers.__version__, "| peft", peft.__version__,
      "| transformers", transformers.__version__, "| torch", torch.__version__)

from huggingface_hub import HfApi
for repo in ["sd-legacy/stable-diffusion-v1-5",
             "stable-diffusion-v1-5/stable-diffusion-v1-5",
             "CompVis/stable-diffusion-v1-4"]:
    try:
        info = HfApi().model_info(repo)
        print("OK     ", repo, "| gated :", info.gated)
    except Exception as e:
        print("ABSENT ", repo, "|", type(e).__name__)

diffusers 0.42.1 | peft 0.21.2 | transformers 5.22.1 | torch 2.14.1+cu130
OK      sd-legacy/stable-diffusion-v1-5 | gated : False
OK      stable-diffusion-v1-5/stable-diffusion-v1-5 | gated : False
OK      CompVis/stable-diffusion-v1-4 | gated : False


In [ ]:
%cd /content/vlm-legers-quantification
!mkdir -p training
!wget -q -O training/train_text_to_image_lora.py https://raw.githubusercontent.com/huggingface/diffusers/v0.42.1/examples/text_to_image/train_text_to_image_lora.py || wget -q -O training/train_text_to_image_lora.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora.py
!ls -l training
!pip install -q ftfy tensorboard Jinja2

/content/vlm-legers-quantification
total 44
-rw-r--r-- 1 root root 43229 Nov 21 16:00 train_text_to_image_lora.py


In [ ]:
!python training/train_text_to_image_lora.py --help | grep -E -- "--(rank|caption_column|train_data_dir|max_train_steps|gradient_checkpointing|learning_rate|seed)"

                                   [--train_data_dir TRAIN_DATA_DIR]
                                   [--caption_column CAPTION_COLUMN]
                                   [--cache_dir CACHE_DIR] [--seed SEED]
                                   [--max_train_steps MAX_TRAIN_STEPS]
                                   [--gradient_checkpointing]
                                   [--learning_rate LEARNING_RATE]
                                   [--noise_offset NOISE_OFFSET] [--rank RANK]
  --train_data_dir TRAIN_DATA_DIR
  --caption_column CAPTION_COLUMN
  --seed SEED           A seed for reproducible training.
  --max_train_steps MAX_TRAIN_STEPS
  --gradient_checkpointing
  --learning_rate LEARNING_RATE
  --rank RANK           The dimension of the LoRA update matrices.


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
import re
from pathlib import Path

chemin = Path("/content/vlm-legers-quantification/training/train_text_to_image_lora.py")
texte = chemin.read_text()

if "CORRECTIF fp32" in texte:
    print("Correctif déjà appliqué")
else:
    m = re.search(r"^([ \t]*)unet\.add_adapter\(unet_lora_config\)[ \t]*$", texte, flags=re.M)
    assert m, "ligne add_adapter introuvable"
    i = m.group(1)  # même indentation que la ligne repérée
    bloc = (f"\n{i}# CORRECTIF fp32 : les poids LoRA entraînables doivent être en float32\n"
            f"{i}for _p in unet.parameters():\n"
            f"{i}    if _p.requires_grad:\n"
            f"{i}        _p.data = _p.data.to(torch.float32)")
    chemin.write_text(texte[:m.end()] + bloc + texte[m.end():])
    print("Correctif appliqué")

!grep -n -A4 "unet.add_adapter(unet_lora_config)" /content/vlm-legers-quantification/training/train_text_to_image_lora.py

Correctif appliqué
545:    unet.add_adapter(unet_lora_config)
546-    # CORRECTIF fp32 : les poids LoRA entraînables doivent être en float32
547-    for _p in unet.parameters():
548-        if _p.requires_grad:
549-            _p.data = _p.data.to(torch.float32)


In [ ]:
%%bash
cd /content/vlm-legers-quantification
nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits -l 5 > /content/vram_log.txt &
SMI_PID=$!
START=$(date +%s)
accelerate launch --mixed_precision="fp16" training/train_text_to_image_lora.py \
  --pretrained_model_name_or_path="sd-legacy/stable-diffusion-v1-5" \
  --train_data_dir="data/raw/lora_train" \
  --caption_column="text" \
  --resolution=512 --random_flip \
  --train_batch_size=1 --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --max_train_steps=250 \
  --learning_rate=1e-04 --lr_scheduler="cosine" --lr_warmup_steps=0 \
  --rank=4 --seed=42 \
  --output_dir="/content/lora_out"
STATUS=$?
END=$(date +%s)
kill $SMI_PID
echo "Durée : $((END-START)) s | code de sortie : $STATUS"
echo "VRAM max (nvidia-smi) : $(sort -n /content/vram_log.txt | tail -1) Mio"

Durée : 507 s | code de sortie : 0
VRAM max (nvidia-smi) : 4363 Mio


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
11/21/2026 16:14:32 - INFO - __main__ - [RANK 0] Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
11/21/2026 16:14:32 - INFO - httpx - HTTP Request: HEAD https://huggingface.co/sd-legacy/stable-diffusion-v1-5/resolve/main/scheduler/scheduler_config.json "HTTP/1.1 307 Temporary Redirect"
11/21/2026 16:14:32 - I

In [ ]:
!ls -l /content/lora_out
!mkdir -p /content/drive/MyDrive/vlm_lora/lora_out
!cp -r /content/lora_out/* /content/drive/MyDrive/vlm_lora/lora_out/

total 3156
drwxr-xr-x 3 root root    4096 Nov 21 16:11 logs
-rw------- 1 root root 3226184 Nov 21 16:22 pytorch_lora_weights.safetensors


In [ ]:
!ls -l /content/drive/MyDrive/vlm_lora/lora_out

total 3155
drwx------ 3 root root    4096 Nov 21 16:11 logs
-rw------- 1 root root 3226184 Nov 21 16:22 pytorch_lora_weights.safetensors
